# 02 · Public-data audit, stage 1: NEON towers — metadata, co-location and access

**Question.** Can a public source supply, at one place, (a) timestamped images, (b) a reference RH/T label and (c) an *independent* conditioning thermometer, with usable licences, before any data are downloaded?

NEON terrestrial towers are the first candidate. Stage 1 (this notebook) uses only metadata APIs: product descriptions, site × month availability of the three products, PhenoCam camera metadata and a data-endpoint access check. Stage 2 (join quality: match counts, timestamp lag distribution, missingness, QC exclusions) needs data files and is planned at the end. Every value below is queried live; the timestamp records when.

In [1]:
import datetime

import pandas as pd
import requests

API, PHENO = 'https://data.neonscience.org/api/v0', 'https://phenocam.nau.edu/api/cameras'
get = lambda url, **kw: requests.get(url, timeout=60, **kw)
pd.set_option('display.max_colwidth', 90)
print('queried at', datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='minutes'))

queried at 2026-10-23T20:08+00:00


In [2]:
PRODUCTS = {'DP1.00098.001': 'label RH + T', 'DP1.00003.001': 'conditioning T', 'DP1.00033.001': 'images'}
meta = {p: get(f'{API}/products/{p}').json()['data'] for p in PRODUCTS}
pd.DataFrame({p: {'role': r, 'name': meta[p]['productName'], 'sensor': meta[p]['productSensor'], 'sites': len(meta[p]['siteCodes']),
                  'design': meta[p]['productDesignDescription'][:220]} for p, r in PRODUCTS.items()}).T

,role,name,sensor,sites,design
DP1.00098.001,label RH + T,Relative humidity,Vaisala HUMICAP Humidity and Temperature Probe - HMP 155,80,"The sensors are installed at the top level of the tower infrastructure, at the soil ar..."
DP1.00003.001,conditioning T,Triple aspirated air temperature,"Thermometrics Climate RTD 100 Ω Probe, housed within a Met One 076B fan aspirated radi...",47,The Triple Aspirated Air Temperature assembly is deployed at core and relocatable towe...
DP1.00033.001,images,Phenology images,Stardot - NetCam SC CAM‐SEC5IR‐B and Live2 cameras,47,NEON has deployed a Stardot cameras on the top of all terrestrial core and gradient to...


The product records state that the HMP155 RH/T probe, the triple fan-aspirated PRT assembly and the phenology camera are all at the **top level of the same tower**, the camera capturing RGB and RGB+NIR every 15 min (30-min images go to the PhenoCam archive). The label unit and the aspirated thermometer are physically different sensors, so `ref_sensor_id ≠ cond_sensor_id` is achievable here. The HMP155 also appears at soil arrays and aquatic stations, so the tower-top position must be selected by its sensor-position code in stage 2.

In [3]:
months = {p: {s['siteCode']: set(s['availableMonths']) for s in meta[p]['siteCodes']} for p in PRODUCTS}
common = sorted(set.intersection(*(set(m) for m in months.values())))
sites = pd.DataFrame(get(f'{API}/sites').json()['data']).set_index('siteCode')
cov = pd.DataFrame({s: (lambda m: {'domain': sites.loc[s, 'domainCode'], 'name': sites.loc[s, 'siteName'], 'state': sites.loc[s, 'stateCode'],
                                   'months_all3': len(m), 'first': min(m), 'last': max(m)})(set.intersection(*(months[p][s] for p in PRODUCTS))) for s in common}).T
print(f'{len(common)} sites carry all three products; months with all three: median {cov.months_all3.median():.0f}, total {cov.months_all3.sum()}')
cov.sort_values('months_all3', ascending=False).head(12)

47 sites carry all three products; months with all three: median 116, total 5260


,domain,name,state,months_all3,first,last
SERC,D02,Smithsonian Environmental Research Center NEON,MD,128,2016-02,2026-09
HEAL,D19,Healy NEON,AK,125,2016-05,2026-09
KONA,D06,Konza Prairie Agroecosystem NEON,KS,125,2016-05,2026-09
CPER,D10,Central Plains Experimental Range NEON,CO,124,2016-06,2026-09
LENO,D08,Lenoir Landing NEON,AL,122,2016-05,2026-09
BLAN,D02,Blandy Experimental Farm NEON,VA,118,2016-12,2026-09
WOOD,D09,Chase Lake National Wildlife Refuge NEON,ND,118,2016-12,2026-09
UNDE,D05,University of Notre Dame Environmental Research Center NEON,MI,118,2016-12,2026-09
ORNL,D07,Oak Ridge NEON,TN,118,2016-12,2026-09
ONAQ,D15,Onaqui NEON,UT,118,2016-12,2026-09


In [4]:
cams = []
for s in common:
    r = get(f"{PHENO}/NEON.{cov.loc[s, 'domain']}.{s}.DP1.00033/", params={'format': 'json'})
    j = r.json() if r.ok else {}
    sm = j.get('sitemetadata') or {}
    cams.append({'site': s, 'status': r.status_code, 'lat': j.get('Lat'), 'lon': j.get('Lon'), 'elev_m': j.get('Elev'), 'utc_offset_h': j.get('utc_offset'),
                 'first': j.get('date_first'), 'last': j.get('date_last'), 'active': j.get('active'), 'facing': sm.get('camera_orientation'), 'veg': sm.get('primary_veg_type')})
cams = pd.DataFrame(cams).set_index('site')
print(cams.status.value_counts().to_dict(), '| facing:', cams.facing.value_counts().to_dict())
cams.join(cov.months_all3).sort_values('months_all3', ascending=False).head(12)

{200: 47} | facing: {'N': 40, 'NNE': 2, 'W': 1}


,status,lat,lon,elev_m,utc_offset_h,first,last,active,facing,veg,months_all3
site,,,,,,,,,,,
SERC,200,38.890080,-76.560010,30,-5,2016-01-29,2026-10-22,True,N,DB,128
HEAL,200,63.875690,-149.213340,409,-9,2016-05-06,2026-10-22,True,N,TN,125
KONA,200,39.110446,-96.612935,443,-6,2016-05-06,2026-10-22,True,N,AG,125
CPER,200,40.815530,-104.745600,1600,-7,2016-06-29,2026-10-22,True,N,GR,124
LENO,200,31.853880,-88.161220,10,-6,2016-05-06,2026-10-22,True,N,DB,122
BLAN,200,39.033698,-78.041788,162,-5,2016-12-14,2026-10-22,True,N,DB,118
WOOD,200,47.128200,-99.241334,585,-6,2016-12-18,2026-10-22,True,N,GR,118
UNDE,200,46.233910,-89.537254,529,-6,2016-12-14,2026-10-22,True,N,DB,118
ORNL,200,35.964128,-84.282588,351,-5,2016-12-15,2026-10-22,True,W,DB,118


PhenoCam reports a fixed `utc_offset` per camera (local standard time, no DST); stage 2 must confirm whether image filenames use that clock or UTC, against the NEON 1-min series (e.g. sunrise brightness vs shortwave radiation).

In [5]:
probe = {p: get(f'{API}/data/{p}/HARV/2023-07') for p in ['DP1.00098.001', 'DP1.00003.001']}
pd.DataFrame({p: {'http_status': r.status_code, 'body': r.text[:80]} for p, r in probe.items()}).T

,http_status,body
DP1.00098.001,403,"{""error"":{""status"":403,""detail"":""Access Denied""},""data"":null}"
DP1.00003.001,403,"{""error"":{""status"":403,""detail"":""Access Denied""},""data"":null}"


**Access blocker.** The NEON data-file endpoint answers HTTP 403 "Access Denied" to these unauthenticated requests, while the metadata endpoints work. Stage 2 therefore needs a NEON API token (free user account; the token goes in an environment variable, never in the repository) or files downloaded manually from the NEON Data Portal into `data/raw/` (ignored by Git). No workaround was attempted.

**Licences.** The PhenoCam fair-use statement (https://phenocam.nau.edu/webcam/fairuse_statement/) places archive imagery and data under CC BY 4.0. The NEON data-policy page (https://www.neonscience.org/data-samples/data-policies-citation) states CC BY 2.0 for website images; the licence of NEON *data products* must be confirmed before redistribution of derived tables.

## Stage-2 plan (needs data access)

1. One site with long coverage and varied humidity (candidates from the coverage table; choose before looking at RH values), one summer and one winter month to start.
2. Label: HMP155 tower-top 1-min RH and T with final quality flags → `td_ref_c`, `d_ref_k` (Magnus) and `d_censored` with the HMP155 datasheet accuracy for the censor limit.
3. Conditioning: triple-aspirated PRT 1-min mean, its own QC; report the HMP155 − PRT temperature difference against shortwave radiation (a direct check of the radiation-error mechanism of notebook 01 §9).
4. Images: PhenoCam RGB frames only (exclude the RGB+NIR alternate), timestamps from filenames converted with the camera `utc_offset`, verified against radiation onset; check for burned-in overlays, camera moves and field-of-view changes.
5. Join report: images per day, fraction with a valid label within ±1 min, lag distribution, missingness by product, QC exclusions with counts, daylight-only counts; no features, no models.